# Degree 2

Basis $\ket{0}, \ket{1}, \ket{2}$.

$$
    \begin{aligned}
        \phi_1 &= (\phi_1)_0h_0+(\phi_1)_1h_1+(\phi_1)_2h_2 \\
        \phi_2 &= (\phi_2)_0h_0+(\phi_2)_1h_1+(\phi_2)_2h_2
    \end{aligned}
$$

In [1]:
from cq.numeric import *

D = 2
n = 10000

v, w = rand_ortho_pair(D+1, n)
T = hermfkin(v) + hermfkin(w)
rho = states_to_density(v, w)
g = rho_to_g(rho)

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

X = np.column_stack([g[:,1:], T]) #remove g_0 to avoid bias ruining implicit polynomial fit
X_train, X_test = train_test_split(X, test_size=0.3)

Finding the lowest degree:

- quadratic in $\vec{g}$
- monic linear in $T$

In [3]:
for d in range(1, 5):
    pows = powers(X.shape[1], d, 1, True)
    f, p = ImplicitPolynomial.fit(X_train, pows)
    p = p[0]
    print('Degree', d, ' std:', np.std(p(X_test)))
    if f:
        break

Degree 1  std: 0.12261977146281286
Degree 2  std: 6.406070740934577e-16


C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\3719730431.py:3: UserWarning: Fit didn't converge: 10.30109085115778!
  f, p = ImplicitPolynomial.fit(X_train, pows)


In [4]:
from cq.pythonic import vecabsq, rand_ortho_pair
from cq.symbolic import hermfkin, rho_to_g
from tqdm.auto import trange

n = pows.shape[0]
g, T = [], []
for _ in trange(n):
    v, w = rand_ortho_pair(D+1, grade=4)
    va, wa = vecabsq(v), vecabsq(w)
    
    T.append((hermfkin(v)/va + hermfkin(w)/wa).expand())
    rho = states_to_density(v)/va + states_to_density(w)/wa
    g.append(tuple(rho_to_g(rho).expand()))

g, T = np.array(g), np.array(T)

  0%|          | 0/16 [00:00<?, ?it/s]

In [5]:
from radicalfield import QuadraticElement235

vtoqe235 = np.vectorize(QuadraticElement235.from_expr, [object])

g, T = vtoqe235(g), vtoqe235(T)
X = np.column_stack([g[:,1:], T])

In [6]:
A = transform(X, pows)
k = dimker(A.astype(float))
assert k == 1 #assert non degenerate sample

In [7]:
from linalg import nullspace

K = nullspace(A, progress=True)

neg:   0%|          | 0/64 [00:00<?, ?it/s]

sub:   0%|          | 0/3840 [00:00<?, ?it/s]

mul:   0%|          | 0/3840 [00:00<?, ?it/s]

truediv:   0%|          | 0/256 [00:00<?, ?it/s]

bool:   0%|          | 0/16 [00:00<?, ?it/s]

gt:   0%|          | 0/240 [00:00<?, ?it/s]

abs:   0%|          | 0/256 [00:00<?, ?it/s]

In [8]:
from sympy import *

syms = symbols(f'g1:{X.shape[1]} T')
for c in K.T:
    tmp = ImplicitPolynomial(pows, c).to_sympy(*syms, trim=False)
    tmp = Poly(tmp, syms[-1])
    display(tmp)
    #print(latex(Poly(tmp, syms[-1]).root(0)))

Poly(T + g1**2/4 - sqrt(6)*g1*g3/3 + g2**2/2 - 2*sqrt(3)*g2*g4/3 - 3*sqrt(2)*g2/4 + 3*g3**2/4 + g4**2 - 1/2, T, domain='EX')

---

Basis $\ket{0}, \ket{1}, \ket{3}$ (skipped $\ket{2}$).

$$
    \begin{aligned}
        \phi_1 &= (\phi_1)_0h_0+(\phi_1)_1h_1+(\phi_1)_2h_3 \\
        \phi_2 &= (\phi_2)_0h_0+(\phi_2)_1h_1+(\phi_2)_2h_3
    \end{aligned}
$$

Can be fitted

- linear in $\vec{g}$
- linear & monic in $T$
- two possibilities (switch & formula)

In [9]:
from cq.numeric import *

n = 10000

v, w = rand_ortho_pair(D+1, n)
v, w = np.insert(v, 2, 0, axis=1), np.insert(w, 2, 0, axis=1)
T = hermfkin(v) + hermfkin(w)
rho = states_to_density(v, w)
g = rho_to_g(rho)

X = np.column_stack([g[:,1:], T])
X_train, X_test = train_test_split(X, test_size=0.3)

pows = powers(X.shape[1], 1, 1, True)
_, p = ImplicitPolynomial.fit(X_train, pows)
assert len(p) == 2
p = p[0]
assert np.isclose(np.std(p(X_test)), 0)

C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\3592026712.py:15: UserWarning: Kernel dimensionality 2; solution is not unique!
  _, p = ImplicitPolynomial.fit(X_train, pows)


In [10]:
from cq.pythonic import rand_ortho_pair
from cq.symbolic import hermfkin, rho_to_g

n = pows.shape[0]
g, T = [], []
for _ in trange(n):
    v, w = rand_ortho_pair(D+1, grade=4)
    v, w = (v[0], v[1], 0, v[2]), (w[0], w[1], 0, w[2])
    va, wa = vecabsq(v), vecabsq(w)
    
    T.append((hermfkin(v)/va + hermfkin(w)/wa).expand())
    rho = states_to_density(v)/va + states_to_density(w)/wa
    g.append(tuple(rho_to_g(rho).expand()))
g, T = np.array(g), np.array(T)

g, T = vtoqe235(g), vtoqe235(T)
X = np.column_stack([g[:,1:], T])

A = transform(X, pows)
k = dimker(A.astype(float))
assert k == 2

syms = symbols(f'g1:{X.shape[1]} T')
K = nullspace(A, progress=True)
for c in K.T:
    tmp = ImplicitPolynomial(pows, c).to_sympy(*syms, trim=False)
    tmp = Poly(tmp, syms[-1])
    display(tmp)
    #print(latex(tmp))
#print(latex(Poly(tmp, syms[-1]).root(0)))

  0%|          | 0/8 [00:00<?, ?it/s]

neg:   0%|          | 0/16 [00:00<?, ?it/s]

sub:   0%|          | 0/448 [00:00<?, ?it/s]

mul:   0%|          | 0/448 [00:00<?, ?it/s]

truediv:   0%|          | 0/64 [00:00<?, ?it/s]

bool:   0%|          | 0/8 [00:00<?, ?it/s]

gt:   0%|          | 0/56 [00:00<?, ?it/s]

abs:   0%|          | 0/64 [00:00<?, ?it/s]

Poly(g5, T, domain='ZZ[g5]')

Poly(T - sqrt(2)*g2/4 + sqrt(6)*g4/4 - 9*sqrt(5)*g6/20 - 1/2, T, domain='EX')

---

Basis $\ket{0}, \ket{2}, \ket{3}$ (skipped $\ket{1}$).

$$
    \begin{aligned}
        \phi_1 &= (\phi_1)_0h_0+(\phi_1)_1h_2+(\phi_1)_2h_3 \\
        \phi_2 &= (\phi_2)_0h_0+(\phi_2)_1h_2+(\phi_2)_2h_3
    \end{aligned}
$$

In [11]:
from cq.numeric import *

n = 10000

v, w = rand_ortho_pair(D+1, n)
v, w = np.insert(v, 1, 0, axis=1), np.insert(w, 1, 0, axis=1)
T = hermfkin(v) + hermfkin(w)
rho = states_to_density(v, w)
g = rho_to_g(rho)

X = np.column_stack([g[:,1:], T])
X_train, X_test = train_test_split(X, test_size=0.3)

_, p = ImplicitPolynomial.fit(X_train, pows)
assert len(p) == 2
p = p[0]
assert np.isclose(np.std(p(X_test)), 0)

C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\645096897.py:14: UserWarning: Kernel dimensionality 2; solution is not unique!
  _, p = ImplicitPolynomial.fit(X_train, pows)


In [12]:
from cq.pythonic import rand_ortho_pair
from cq.symbolic import hermfkin, rho_to_g

n = pows.shape[0]
g, T = [], []
for _ in trange(n):
    v, w = rand_ortho_pair(D+1, grade=4)
    v, w = (v[0], 0, v[1], v[2]), (w[0], 0, w[1], w[2])
    va, wa = vecabsq(v), vecabsq(w)
    
    T.append((hermfkin(v)/va + hermfkin(w)/wa).expand())
    rho = states_to_density(v)/va + states_to_density(w)/wa
    g.append(tuple(rho_to_g(rho).expand()))
g, T = np.array(g), np.array(T)

g, T = vtoqe235(g), vtoqe235(T)
X = np.column_stack([g[:,1:], T])

A = transform(X, pows)
k = dimker(A.astype(float))
assert k == 2

K = nullspace(A, progress=True)
for c in K.T:
    tmp = ImplicitPolynomial(pows, c).to_sympy(*syms, trim=False)
    tmp = Poly(tmp, syms[-1])
    display(tmp)
#print(latex(Poly(tmp, syms[-1]).root(0)))

  0%|          | 0/8 [00:00<?, ?it/s]

neg:   0%|          | 0/16 [00:00<?, ?it/s]

sub:   0%|          | 0/448 [00:00<?, ?it/s]

mul:   0%|          | 0/448 [00:00<?, ?it/s]

truediv:   0%|          | 0/64 [00:00<?, ?it/s]

bool:   0%|          | 0/8 [00:00<?, ?it/s]

gt:   0%|          | 0/56 [00:00<?, ?it/s]

abs:   0%|          | 0/64 [00:00<?, ?it/s]

Poly(-sqrt(30)*g1/3 + g5, T, domain='EX')

Poly(T + sqrt(2)*g2/4 - sqrt(6)*g4/3 + 3*sqrt(5)*g6/10 - 1/2, T, domain='EX')

---

Basis $\ket{1}, \ket{2}, \ket{3}$ (skipped $\ket{0}$).

$$
    \begin{aligned}
        \phi_1 &= (\phi_1)_0h_1+(\phi_1)_1h_2+(\phi_1)_2h_3 \\
        \phi_2 &= (\phi_2)_0h_1+(\phi_2)_1h_2+(\phi_2)_2h_3
    \end{aligned}
$$

In [13]:
from cq.numeric import *

n = 10000

v, w = rand_ortho_pair(D+1, n)
v, w = np.insert(v, 0, 0, axis=1), np.insert(w, 0, 0, axis=1)
T = hermfkin(v) + hermfkin(w)
rho = states_to_density(v, w)
g = rho_to_g(rho)

X = np.column_stack([g[:,1:], T])
X_train, X_test = train_test_split(X, test_size=0.3)

_, p = ImplicitPolynomial.fit(X_train, pows)
assert len(p) == 2
p = p[0]
assert np.isclose(np.std(p(X_test)), 0)

C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\3871484834.py:14: UserWarning: Kernel dimensionality 2; solution is not unique!
  _, p = ImplicitPolynomial.fit(X_train, pows)


In [14]:
from cq.pythonic import rand_ortho_pair
from cq.symbolic import hermfkin, rho_to_g

n = pows.shape[0]
g, T = [], []
for _ in trange(n):
    v, w = rand_ortho_pair(D+1, grade=4)
    v, w = (0, v[0], v[1], v[2]), (0, w[0], w[1], w[2])
    va, wa = vecabsq(v), vecabsq(w)
    
    T.append((hermfkin(v)/va + hermfkin(w)/wa).expand())
    rho = states_to_density(v)/va + states_to_density(w)/wa
    g.append(tuple(rho_to_g(rho).expand()))
g, T = np.array(g), np.array(T)

g, T = vtoqe235(g), vtoqe235(T)
X = np.column_stack([g[:,1:], T])

A = transform(X, pows)
k = dimker(A.astype(float))
assert k == 2

K = nullspace(A, progress=True)
for c in K.T:
    tmp = ImplicitPolynomial(pows, c).to_sympy(*syms, trim=False)
    tmp = Poly(tmp, syms[-1])
    display(tmp)
    #print(latex(tmp))
#print(latex(Poly(tmp, syms[-1]).root(0)))

  0%|          | 0/8 [00:00<?, ?it/s]

neg:   0%|          | 0/16 [00:00<?, ?it/s]

sub:   0%|          | 0/448 [00:00<?, ?it/s]

mul:   0%|          | 0/448 [00:00<?, ?it/s]

truediv:   0%|          | 0/64 [00:00<?, ?it/s]

bool:   0%|          | 0/8 [00:00<?, ?it/s]

gt:   0%|          | 0/56 [00:00<?, ?it/s]

abs:   0%|          | 0/64 [00:00<?, ?it/s]

Poly(sqrt(30)*g1/3 - 2*sqrt(5)*g3/3 + g5, T, domain='EX')

Poly(T + 5*sqrt(2)*g2/4 - sqrt(6)*g4/2 + 3*sqrt(5)*g6/10 - 13/2, T, domain='EX')

---

Fitting

- two particles
- in $(\ket{0}, \ket{1}, \ket{2})$
- that aren't orthonormal.

Lowest degrees found:

- 3 in $\vec{g}$
- 3 & monic in $T$.

Fit is unique.

In [15]:
from cq.numeric import *

n = 10000

v, w = 2*np.random.rand(2, n, D+1) - 1
T = hermfkin(v) + hermfkin(w)
rho = states_to_density(v, w)
g = rho_to_g(rho)

X = np.column_stack([g, T])
X_train, X_test = train_test_split(X, test_size=0.3)

for d in range(1, 5):
    pows = powers(X.shape[1], d, None, True)
    f, p = ImplicitPolynomial.fit(X_train, pows)
    p = p[0]
    print('Degree', d, ' std:', np.std(p(X_test)))
    if f:
        break

C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\3512681860.py:15: UserWarning: Fit didn't converge: 16.72268191134439!
  f, p = ImplicitPolynomial.fit(X_train, pows)
C:\Users\S.Goessl\AppData\Local\Temp\ipykernel_31008\3512681860.py:15: UserWarning: Fit didn't converge: 3.1586720300566182!
  f, p = ImplicitPolynomial.fit(X_train, pows)


Degree 1  std: 0.19788266528650342
Degree 2  std: 0.03692041956204068
Degree 3  std: 7.459520975805444e-15


In [16]:
from cq.pythonic import vrandq
from cq.symbolic import hermfkin, rho_to_g

n = pows.shape[0]
g, T = [], []
for _ in trange(n):
    v, w = vrandq(D+1, grade=4), vrandq(D+1, grade=4)
    T.append((hermfkin(v) + hermfkin(w)).expand())
    rho = states_to_density(v) + states_to_density(w)
    g.append(tuple(rho_to_g(rho).expand()))
g, T = np.array(g), np.array(T)

g, T = vtoqe235(g), vtoqe235(T)
X = np.column_stack([g, T])

A = transform(X, pows)
k = dimker(A.astype(float))
assert k == 1

K = nullspace(A, progress=True)
syms = symbols(f'g0:{X.shape[1]-1} T')
for c in K.T:
    tmp = ImplicitPolynomial(pows, c).to_sympy(*syms, trim=False)
    tmp = Poly(tmp, syms[-1])
    display(tmp)
    #print(latex(tmp))

  0%|          | 0/84 [00:00<?, ?it/s]

neg:   0%|          | 0/1764 [00:00<?, ?it/s]

sub:   0%|          | 0/585648 [00:00<?, ?it/s]

mul:   0%|          | 0/585648 [00:00<?, ?it/s]

truediv:   0%|          | 0/7056 [00:00<?, ?it/s]

bool:   0%|          | 0/84 [00:00<?, ?it/s]

gt:   0%|          | 0/6972 [00:00<?, ?it/s]

abs:   0%|          | 0/7056 [00:00<?, ?it/s]

Poly(T**3 + (-3*g0/4 - sqrt(2)*g2/4)*T**2 + (3*g0**2/16 + sqrt(2)*g0*g2/8 - sqrt(6)*g0*g4/3 + sqrt(6)*g1*g3/6 - g2**2/8 + 2*sqrt(3)*g2*g4/3 - g3**2/2 - g4**2)*T - g0**3/64 - sqrt(2)*g0**2*g2/64 + sqrt(6)*g0**2*g4/12 - sqrt(6)*g0*g1*g3/24 + g0*g2**2/32 - sqrt(3)*g0*g2*g4/3 + 7*g0*g3**2/24 + 11*g0*g4**2/12 + sqrt(6)*g1**2*g4/12 - sqrt(3)*g1*g2*g3/12 - g1*g3*g4/3 + sqrt(2)*g2**3/32 + sqrt(2)*g2*g3**2/24 - sqrt(2)*g2*g4**2/4 + sqrt(6)*g3**2*g4/12 + sqrt(6)*g4**3/9, T, domain='EX')